## Проект: Retrieval-Augmented Generation (RAG) на данных StackOverflow
## Описание проекта

Цель проекта — построить систему поиска и генерации ответов на основе данных StackOverflow с использованием подхода Retrieval-Augmented Generation (RAG).

В рамках проекта реализован полный пайплайн подготовки данных, разбиения документов на чанки, генерации эмбеддингов и хранения их в векторной базе данных PostgreSQL с расширением pgvector.

Подход RAG позволяет находить релевантные фрагменты текста с помощью векторного поиска и использовать их в качестве контекста для генерации ответа языковой моделью.

In [ ]:
from src.core.config import Settings
from src.pipeline_loader import LoaderPipeline
from src.pipeline_retrieval import RetrievalPipeline
import psycopg2
from src.indexing.embedder import EmbeddingGenerator
from src.reranker.reranker import Reranker
from src.core.logging import setup_logging
from src.core.queries import INSERT_CHUNK_QUERY, INSERT_CHUNK_QUERY_TEST

import logging
settings = Settings()
embedding_generator = EmbeddingGenerator()
encoder = Reranker(settings)

setup_logging(settings)

logger = logging.getLogger(__name__)
logger.info("Логгер запущен")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-26 09:38:38,818 | INFO | __main__ | Логгер запущен


In [2]:
pipeline_loader = LoaderPipeline(settings, embedding_generator)
pipeline_retrieval = RetrievalPipeline(settings, embedding_generator, encoder)

2026-03-26 09:38:38,832 | INFO | src.retrieval.reranker | Загрузка reranker-модели: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-26 09:38:41,196 | INFO | src.generation.llm_client | Загрузка LLM-модели: google/flan-t5-large


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
2026-03-26 09:38:47,322 | INFO | src.generation.llm_client | LLM загружена на устройство: cuda


In [ ]:
data = pipeline_loader.run()

with psycopg2.connect(**settings.DB_PARAMS) as conn:
    with conn.cursor() as cursor:
        rows = pipeline_loader.db_saver.build_insert_rows(data)
        pipeline_loader.db_saver.insert_rows(rows, conn, cursor, INSERT_CHUNK_QUERY_TEST)

### 6. Поиск

In [3]:
query = "How to print Hello World in Python?"
answer = pipeline_retrieval.run(query)

2026-03-26 09:38:47,339 | INFO | src.pipeline_retrieval | Шаг 1: Формирование запроса в базу данных.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-26 09:38:47,545 | INFO | src.pipeline_retrieval | Эмбеддинг запроса [0.019435974, 0.08953558, -0.04974294, 0.015652642, -0.026656052, -0.15823485, -0.018088067, -0.01309944, -0.10564635, -0.017737221, 0.08653759, -0.038171068, 0.036263745, -0.002315856, 0.030325005, -0.05694112, -0.038167913, 0.0004744887, 0.03448758, -0.09435223, 0.054428585, 0.07680798, 0.066108495, 0.033949286, -0.005945955, -0.038748406, 0.021388156, 0.09014582, 0.010623463, 0.0031868836, 0.02249712, 0.02411088, 0.13482465, 0.00031163244, 0.08159091, 0.06244769, 0.018858595, -0.066973194, -0.020442745, 0.010359467, 0.047454935, -0.009492643, -0.0372886, -0.011283675, -0.06860958, 0.002335099, -0.13637754, 0.063536495, 0.07791794, 0.03873432, -0.036505617, -0.036837187, -0.040071495, -0.058308356, 0.08262671, 0.026748298, 0.040195115, 0.00022049417, -0.002612344, -0.099448204, -0.08233489, 0.025788054, -0.034292474, -0.006652218, 0.08424411, -0.033406913, -0.03525175, 0.08554044, -0.034833357, -0.042910498, 

Чанки [{'chunk_id': 'q10395430_a10395681_c4', 'chunk_text': "print `` Hello World ''", 'distance': 0.12231981754302979, 'question_id': 10395430, 'answer_id': 10395681, 'chunk_index': 4, 'title': 'Are Vala and Genie production ready?', 'tags': 'c, genie, gobject, vala'}, {'chunk_id': 'q6953550_a6953626_c0', 'chunk_text': "Title : Why do ` print ( `` Hello , World ! `` ) ` and ` print ( `` Hello '' , `` World ! `` ) ` produce different outputs ? Tags : python Question : I just installed Python 2.7.2 on Windows XP with the idea of learning how to program . Several of the tutorial books I 'm using give examples of print commands which , when I try them , I get different answers . I expected both of these to return the same thing - > > > print ( `` Hello , World ! '' ) Hello , World ! > > > print ( `` Hello '' , `` World '' ) ( 'Hello ' , 'World ' ) > > > I 've tried searching around for answers , but I 'm not even sure how to explain where I 'm", 'distance': 0.26851005962550867, 'question_

2026-03-26 09:39:03,895 | INFO | src.retrieval.reranker | После rerank оставляем top-1
2026-03-26 09:39:03,896 | INFO | src.pipeline_retrieval | Всего чанков: 1
2026-03-26 09:39:03,897 | INFO | src.pipeline_retrieval | Шаг 4: Сборка контекста
2026-03-26 09:39:03,897 | INFO | src.retrieval.prompt_builder | Начало build_context. Получено чанков: 1
2026-03-26 09:39:03,899 | INFO | src.retrieval.prompt_builder | Собран context длиной 465 символов
2026-03-26 09:39:03,899 | INFO | src.pipeline_retrieval | Шаг 5: Сборка prompt
2026-03-26 09:39:03,901 | INFO | src.pipeline_retrieval | Шаг 6: Генерация ответа


Реранкер [{'chunk_id': 'q17132740_a17132766_c4', 'chunk_text': "longer a statement , but a function . Thus , to use it , you would do print ( ) instead of print . Python 2.7 : print `` Hello world ! '' Python 3 : print ( `` Hello world ! '' ) You can read more about the print ( ) function at the docs , which also mentions some parameters you can include , and also how you can use it in python 2.x ( from __future__ import print_function ) Also , you 're missing a colon after if o1 == '' T '' . May just be a simple typo : ) .", 'distance': 0.3032058892723376, 'question_id': 17132740, 'answer_id': 17132766, 'chunk_index': 4, 'title': 'What is wrong with my code? I am new at Python 3.3', 'tags': 'python, python-3.x, syntax, syntax-error, text-based', 'rerank_score': 8.281172752380371}]
Контекст longer a statement , but a function . Thus , to use it , you would do print ( ) instead of print . Python 2.7 : print `` Hello world ! '' Python 3 : print ( `` Hello world ! '' ) You can read more a

2026-03-26 09:39:04,892 | INFO | src.generation.llm_client | Сгенерирован ответ длиной 26 символов
2026-03-26 09:39:04,893 | INFO | src.pipeline_retrieval | Pipeline завершён
2026-03-26 09:39:04,894 | INFO | src.core.config | Время выполнения функции 'run': 17.5551 секунд


Ответ print('' Hello world ! '')


In [5]:
print(answer)

Hello and welcome to my world '' ) print print (  I 'd like you to meet my family
